# Bike Availability Forecasting

This notebook trains a regression models to forecast  bike stocks at stations using temporal lags, weather, calendar variables and tensor-based encodings.

The pipeline includes:
- Data loading
- Tensor decomposition based feature encoding
- Covariate construction
- Model training and evaluation


==============================
Selected Model:
We selected Xgboost as the most suitable model in practice (zero preprocessing, zero hyperparameter tuning, fast computing)

**Forecasting Framework: The Envelope Method**
Instead of predicting a single bike stock outcome, we forecast stocks across **4 extreme rebalancing scenarios** (Envelopes).

These extreme scenarios form a basis that allows for the reconstruction of **any** intermediate rebalancing policy using the *Envelope Reconstruction Formula*.

 Here we show how to predict the reference scenario ("bike_0") wich is just the stock given current rebalancing strategie, but one should also predict MIN, MAX, min and max scenarios defined in previous section (envelope_reconstruction)

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Tensor decomposition
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly import tucker_to_tensor
import xgboost as xgb

SELECTED MODEL:

In [ ]:
model_params = {
    "model": xgb.XGBRegressor,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 8
}


LOAD DATA

In [ ]:
# Load dataset
ds = xr.open_dataset("../full_db.nc")

# Scenario dataset (should contain (fictive) extremal rebalancing policy scenarios
# (MIN, MAX, min & max)
#here only bike_0 scenario is predicted
sc = ds.copy()

#vm = mechanical, vae = E-bike
sc["bike_0"] = ds["vm_disp"] + ds["vae_disp"]

print(ds)

<xarray.Dataset> Size: 330MB
Dimensions:          (binary_index: 11, time: 17544, ferie_feature: 2,
                      spatial_feature: 4, station: 1551, meteo_dim: 8,
                      component: 10)
Coordinates:
  * binary_index     (binary_index) int32 44B 0 1 2 3 4 5 6 7 8 9 10
  * time             (time) datetime64[ns] 140kB 2023-01-01 ... 2024-12-31T23...
  * ferie_feature    (ferie_feature) <U8 64B 'vacances' 'is_ferie'
  * spatial_feature  (spatial_feature) <U12 192B 'Capacité de ' ... 'altitude'
  * station          (station) <U12 74kB '000000002' '10001' ... '9999980'
  * meteo_dim        (meteo_dim) <U5 160B 'DRR1' 'FXI' 'GLO' ... 'TX' 'U'
  * component        (component) int64 80B 0 1 2 3 4 5 6 7 8 9
    capacite         (station) int32 6kB ...
    latitude         (station) float64 12kB ...
    longitude        (station) float64 12kB ...
Data variables: (12/19)
    code_binaire     (station, binary_index) int32 68kB ...
    features         (station, spatial_feature

PIPELINE CONFIGURATION

In [ ]:
horizons = [0, 24, 24 * 7]
nsplits = 4
scenarios = ["0"] #+ MIN, MAX, min, max]
fill_db = True

pipeline should run in < 5 minutes


TIME CONFIGURATION

In [5]:
# First Monday of 2023 (dataset starts on a Sunday)
first_monday_index = 24

n_stations = len(sc["station"])

n_time_full = len(sc["time"]) - first_monday_index

n_weeks = n_time_full // (7 * 24)

n_time = 24 * 7 * n_weeks
n_days = 7 * n_weeks

CAPACITY PREPROCESSING

In [6]:
#WARNING ! data contain a few "negative capacity station" due to a preprocesing bug
# Avoid degenerate stations
sc["capacite"][sc["capacite"] <= 5] = 0

capacity = np.array(sc["capacite"])

UTILITY: Xarray -> Numpy

In [7]:
def to_numpy(variable, dataset=sc, time_mode="right"):
    """
    Convert xarray variable to numpy array with optional weekly reshaping.
    """

    if not time_mode:
        return np.array(dataset[variable])

    da = dataset[variable]

    if time_mode == "right":
        return np.array(da)[:, first_monday_index:first_monday_index + n_weeks*24*7]\
            .reshape(da.shape[0], n_weeks, 7*24)

    if time_mode == "left":
        return np.array(da)[first_monday_index:first_monday_index + n_weeks*24*7, :]\
            .reshape(n_weeks, 7*24, da.shape[1])

UTILITY: Some reshaping staff

In [8]:
def broadcast_to_shape(x, shape):
    """
    Broadcast tensor to a target shape (except last dimension).
    """

    if shape is None:
        return x

    target = []

    for xd, sd in zip(x.shape[:-1], shape):

        if xd == sd or xd == 1:
            target.append(sd)
        else:
            raise ValueError("Incompatible axis")

    target.append(x.shape[-1])

    return np.broadcast_to(x, target)


def concat_features(arrays, shape=None, flatten=False):
    """
    Concatenate features along the last axis with broadcasting.
    """

    if not flatten:
        return np.concatenate(
            [broadcast_to_shape(x, shape) for x in arrays],
            axis=-1
        )

    out = concat_features(arrays, shape=shape)

    dim1 = np.prod(out.shape[:-1])
    dim2 = out.shape[-1]

    return out.reshape(dim1, dim2)

IMPORTANT: EMBEDDING FUNCTION (with Tucker)

In [9]:
rank = [10, n_weeks, 7, 10]

def tensor_encoding(x):
    """
    Extract spatial and temporal embeddings using Tucker decomposition.

    Dimensions:
    station × week × day × hour
    """

    _, factors = tucker(
        x.reshape(n_stations, n_weeks, 7, 24),
        rank=rank
    )

    spatial_encoding = factors[0] / (capacity[:, None] + 0.01)

    day_encoding = factors[2]
    hour_encoding = factors[3]

    time_encoding = concat_features(
        [day_encoding[:, None, :], hour_encoding[None, :, :]],
        shape=[7, 24],
        flatten=True
    )

    return concat_features([capacity[:, None], spatial_encoding]), time_encoding

MODEL FIT

In [ ]:
def fit_model(train_test):
    """
    Train model defined in MODEL_REGISTRY and return predictions.
    """

    X_train, X_test, y_train, y_test = train_test
    X_train = X_train.copy()
    X_test = X_test.copy()
    # -----------------------------------------------------
    # Model instantiation
    # -----------------------------------------------------

    model = model_params["model"](**model_params["params"])

    # -----------------------------------------------------
    # Training
    # -----------------------------------------------------

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    rmse = np.sqrt(np.mean((pred - y_test) ** 2))

    print(f"model RMSE:", rmse)

    return pred

## Model Training

We iterate over:

- scenarios
- cross-validation splits
- forecasting horizons

In [11]:
def transpose_prediction_bugfix(x):
    """
    Workaround for numpy dimension ordering during prediction storage.
    """
    return np.transpose(x, (1, 0, 2))

## Covariates used in the model

- Lagged bike counts
- Meteorological variables
- Holiday indicators
- Spatial station embeddings
- Temporal embeddings (day/hour)
- Weekly seasonal encoding
- Last known daily update

In [12]:
# ---------------------------------------------------------
# Cross-validation split across weeks
# ---------------------------------------------------------

# First 10 weeks cannot be used (lags require history)
valid_weeks = np.arange(10, n_weeks)

# 4-fold temporal split
fold_index = (valid_weeks - 10) % 4

weeks_split = [
    [valid_weeks[fold_index != i], valid_weeks[fold_index == i]]
    for i in range(4)
]


# ---------------------------------------------------------
# Sampling utility for large datasets
# ---------------------------------------------------------

train_sample_size = 3_000_000

def random_sample(x):
    return np.random.choice(x, size=train_sample_size)


# ---------------------------------------------------------
# Static covariates
# ---------------------------------------------------------

lat = to_numpy("latitude", dataset=ds, time_mode=False)
lon = to_numpy("longitude", dataset=ds, time_mode=False)

Meteo = to_numpy("meteo_h", dataset=ds, time_mode="left")
Holiday = to_numpy("feries", dataset=ds, time_mode="left")


# ---------------------------------------------------------
# Seasonal week encoding
# ---------------------------------------------------------

def week_encoding(week_index):
    """
    Encode week index using Fourier seasonal features.
    """
    harmonics = []

    for h in [1, 2, 3]:
        harmonics.append(np.sin(2 * np.pi * h * week_index / 52))
        harmonics.append(np.cos(2 * np.pi * h * week_index / 52))

    return np.column_stack([week_index, *harmonics])


# ---------------------------------------------------------
# Lag utilities
# ---------------------------------------------------------

def lag(array, lag_value):
    """
    Temporal lag operator.
    """
    return np.roll(array, shift=lag_value)


# Weekly historical lags
week_lags = [24 * 7 * i for i in range(1, 10)]


# ---------------------------------------------------------
# Last observed daily value
# ---------------------------------------------------------

def last_daily_update(Y, horizon):
    """
    Last known station value at 23:00 before prediction horizon.
    """

    daily_last = (
        Y.reshape(n_stations, n_weeks, 7, 24)[:, :, :, -1]
        .reshape(n_stations, n_weeks, 7)
    )

    return lag(daily_last, horizon // 24 + 1)

In [13]:
def prepare_data(variable):
    """
    Prepare base tensors and embeddings for a given variable.
    """

    Y = to_numpy(variable)

    spatial_features, temporal_features = tensor_encoding(Y)

    spatial_features = np.concatenate(
        [spatial_features, lat[:, None], lon[:, None]],
        axis=1
    )

    return {
        "Y": Y,
        "space": spatial_features,
        "time": temporal_features
    }

PREPARE DATA FOR REGRESSION TASK

In [14]:
def prepare_split_horizon(split_id, horizon, data_dict):

    Y = data_dict["Y"]
    space = data_dict["space"]
    time = data_dict["time"]

    # Random training sampling
    train_station = random_sample(n_stations)
    train_tow = random_sample(24 * 7)
    train_week = random_sample(weeks_split[split_id][0])

    test_week = weeks_split[split_id][1]
    n_test_weeks = len(test_week)

    # Lag configuration
    lags = [24 * i + horizon for i in range(1, 10)] + week_lags

    # -----------------------------------------------------
    # Target variable
    # -----------------------------------------------------

    Y_train = Y[train_station, train_week, train_tow]

    Y_train_lag = concat_features([
        lag(Y, l)[train_station, train_week, train_tow][:, None]
        for l in lags
    ])

    Y_test = Y[:, test_week, :].reshape(-1)

    Y_test_lag = concat_features([
        lag(Y, l)[:, test_week, :, None]
        for l in lags
    ])

    # -----------------------------------------------------
    # Last daily update
    # -----------------------------------------------------

    update = last_daily_update(Y, horizon)

    update_train = update[
        train_station,
        train_week,
        train_tow // 24
    ][:, None]

    update_test = (
        np.repeat(
            update[:, test_week, :][:, :, :, None],
            24,
            axis=-1
        )
        .reshape(n_stations, n_test_weeks, 24 * 7, 1)
    )

    # -----------------------------------------------------
    # Feature matrix construction
    # -----------------------------------------------------

    X_train = concat_features([
        Y_train_lag,
        Meteo[train_week, train_tow],
        Holiday[train_week, train_tow],
        space[train_station],
        time[train_tow],
        week_encoding(train_week),
        update_train
    ])

    X_test = concat_features([
        Y_test_lag,
        Meteo[test_week][None, :, :, :],
        Holiday[test_week][None, :, :, :],
        space[:, None, None, :],
        time[None, None, :, :],
        week_encoding(test_week)[None, :, None, :],
        update_test
    ],
    shape=[n_stations, n_test_weeks, 24 * 7],
    flatten=True)

    return (X_train, X_test, Y_train, Y_test), test_week

TRAINING LOOP

In [20]:
print(train_sample_size)

3000000


In [ ]:
all_predictions = np.zeros(
    (len(scenarios), 3, n_stations, n_weeks, 24 * 7),
    dtype=np.uint8
)


for scenario_id, scenario in enumerate(scenarios):

    data_dict = prepare_data(scenario)

    for split_id in range(nsplits):

        for horizon_id, horizon in enumerate(horizons):

            print("horizon:", horizon)

            train_test, weeks_pred = prepare_split_horizon(
                split_id,
                horizon,
                data_dict
            )

            pred = fit_model(train_test)


            #saving results in suitable format
            pred = np.clip(pred, 0, 255) \
                .astype(np.uint8) \
                .reshape(
                    n_stations,
                    len(weeks_pred),
                    24 * 7
                )

            pred = transpose_prediction_bugfix(pred)

            all_predictions[
                scenario_id,
                horizon_id,
                :,
                weeks_pred
            ] = pred

horizon: 0
model: xgb
xgb RMSE: 4.760561
model: lin
lin RMSE: 5.2379910850855085
model: ridge
ridge RMSE: 5.237987532773427
model: poly


/home/ROCQ/mathnet/adecacqu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.900e+05, tolerance: 3.469e+02
  model = cd_fast.enet_coordinate_descent(


poly RMSE: 5.065309693572797
model: neural


/home/ROCQ/mathnet/adecacqu/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


neural RMSE: 4.694515779783308
horizon: 24
model: xgb
xgb RMSE: 5.632209
model: lin
lin RMSE: 5.814594483668096
model: ridge
ridge RMSE: 5.814591184695613
model: poly


...

neural RMSE: 5.61017625199355
horizon: 168
model: xgb
xgb RMSE: 5.957616
model: lin
lin RMSE: 6.067969604945067
model: ridge
ridge RMSE: 6.067972319561664
model: poly


/home/ROCQ/mathnet/adecacqu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.529e+05, tolerance: 3.541e+02
  model = cd_fast.enet_coordinate_descent(


poly RMSE: 6.210814724209119
model: neural


/home/ROCQ/mathnet/adecacqu/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


neural RMSE: 5.995965748210053


EXPORT RESULTS

In [ ]:
# ---------------------------------------------------------
# Build prediction dataset
# ---------------------------------------------------------

result = xr.Dataset(
    data_vars={
        "predictions": (
            ["scenario", "horizon", "station", "week", "tow"],
            all_predictions,
        )
    },

    coords={
        "scenario": scenarios,
        "horizon": horizons,
        "station": ds.station,
        "week": np.arange(n_weeks),
        "tow": np.arange(24 * 7),
    }
)


result.attrs.update({

    "title": "Bike availability forecasts under multiple scenarios",

    "summary": (
        "Bike availability predictions generated with machine learning "
        "models trained on historical usage, weather variables and "
        "tensor-based spatial/temporal embeddings."
    ),

    "model_family": "tree-based regression / baseline models",

})

SAVE

In [ ]:
result.to_netcdf("../results/scenarios_predicted.nc")